# Notebook 4: Model Interpretability — SHAP, Biomarker Prioritization, and Biological Meaning

**From Inference to Prediction** | Bioinformatics Big Data Analysis

---

This notebook focuses on the critical gap between **high predictive accuracy** and **biological interpretability**:

1. **Train** an interpretable classification model on simulated omics data
2. **Explain** model predictions using SHAP values
3. **Compare** SHAP ranking with statistical differential expression
4. **Prioritize** candidate biomarkers for follow-up validation
5. **Connect** predictive features back to biological pathways

### Why Interpretability Matters
In biomedical ML, a model is not useful simply because it predicts well. Clinicians and biologists need to know:
- Which genes drive a prediction?
- Are those genes plausible disease mechanisms?
- Do predictive genes overlap with statistically significant genes?

This is the point where machine learning must reconnect with biostatistics.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from scipy.stats import ttest_ind

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
sns.set_style('whitegrid')
np.random.seed(42)

print('Environment ready.')

## 1. Simulate Biomarker Discovery Data

We simulate a binary cancer-vs-normal expression dataset with three gene groups:
- **True driver genes**: biologically causal and strongly predictive
- **Passenger genes**: statistically shifted but weak predictors
- **Interaction genes**: individually modest, but highly predictive in combination

This setup illustrates a common bioinformatics result: **low p-value does not always imply high predictive value**, and vice versa.

In [ ]:
def simulate_biomarker_dataset(n_samples=400, n_genes=120, seed=42):
    rng = np.random.default_rng(seed)
    labels = np.array([1] * (n_samples // 2) + [0] * (n_samples // 2))
    X = rng.normal(0, 1, size=(n_samples, n_genes))
    
    # Strong differential genes
    X[labels == 1, 0:10] += 2.0
    
    # Mild but significant passengers
    X[labels == 1, 10:20] += 0.6
    
    # Interaction-only genes: predictive together, weak individually
    interaction_mask = (X[:, 20] > 0.5) & (X[:, 21] < -0.5)
    X[interaction_mask & (labels == 1), 22:25] += 1.5
    
    gene_names = [f'Gene_{i:03d}' for i in range(n_genes)]
    sample_ids = [f'Sample_{i:03d}' for i in range(n_samples)]
    
    X_df = pd.DataFrame(X, index=sample_ids, columns=gene_names)
    y = pd.Series(labels, index=sample_ids, name='Cancer')
    return X_df, y


X, y = simulate_biomarker_dataset()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Positive class ratio: {y.mean():.1%}')

## 2. Train a Predictive Model

In [ ]:
rf = RandomForestClassifier(
    n_estimators=500,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    oob_score=True,
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {acc:.3f}')
print(f'OOB score:      {rf.oob_score_:.3f}')

## 3. SHAP Feature Attribution

SHAP decomposes each prediction as:

$$f(x) = athbb{E}[f(x)] + um_{j=1}^{p} hi_j$$

where $hi_j$ is the Shapley contribution of feature $j$. Averaging $|hi_j|$ across samples yields a global importance ranking that respects **non-linearity** and **feature interactions**.

In [ ]:
from src.features.selection import shap_feature_ranking

shap_rank = shap_feature_ranking(rf, X_test, top_k=20, model_type='tree')
shap_rank.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
top10 = shap_rank.head(10).iloc[::-1]
ax.barh(top10['gene'], top10['mean_abs_shap'], color='darkslateblue', alpha=0.85)
ax.set_xlabel('Mean |SHAP value|', fontsize=12)
ax.set_ylabel('Gene', fontsize=12)
ax.set_title('Top 10 Predictive Genes by SHAP', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Compare SHAP Ranking with Differential Expression

A classical biostatistics workflow would rank genes by p-value from a two-sample t-test. We compare that ranking to SHAP to show that **significance** and **predictive importance** are related but not identical concepts.

In [ ]:
group1 = X_train[y_train == 1]
group0 = X_train[y_train == 0]

t_stat, p_vals = ttest_ind(group1, group0, axis=0, equal_var=False)
mean_diff = group1.mean(axis=0) - group0.mean(axis=0)

de_table = pd.DataFrame({
    'gene': X_train.columns,
    'p_value': p_vals,
    'abs_mean_diff': np.abs(mean_diff.values),
}).sort_values('p_value')

comparison = shap_rank.merge(de_table[['gene', 'p_value', 'abs_mean_diff']], on='gene', how='left')
comparison.head(15)

In [ ]:
# Scatter: statistical significance vs predictive importance
merged = de_table.merge(shap_rank, on='gene', how='left').fillna({'mean_abs_shap': 0})
merged['neg_log10_p'] = -np.log10(merged['p_value'] + 1e-300)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(merged['neg_log10_p'], merged['mean_abs_shap'], alpha=0.65, color='teal', edgecolors='white', linewidth=0.3)

# Annotate a few interesting genes
interesting = merged.sort_values(['mean_abs_shap', 'neg_log10_p'], ascending=False).head(8)
for _, row in interesting.iterrows():
    ax.text(row['neg_log10_p'] + 0.05, row['mean_abs_shap'] + 0.001, row['gene'], fontsize=9)

ax.set_xlabel('-log10(p-value)', fontsize=12)
ax.set_ylabel('Mean |SHAP|', fontsize=12)
ax.set_title('Statistical Significance vs Predictive Importance', fontsize=13, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Biomarker Prioritization Table

A practical project outcome is not just a model, but a **prioritized short-list** of candidate biomarkers for experimental validation. We combine predictive impact and effect size into a simple prioritization score.

In [ ]:
priority = comparison.copy()
priority['priority_score'] = (
    priority['mean_abs_shap'].rank(pct=True) * 0.7 +
    priority['abs_mean_diff'].rank(pct=True) * 0.3
)
priority = priority.sort_values('priority_score', ascending=False)
priority.head(15)

## 6. Biological Interpretation Template

A strong ML-for-biology project should end with a biologically meaningful narrative, for example:

- Genes ranked highly by both SHAP and t-test are robust biomarker candidates.
- Genes ranked highly by SHAP but not by p-value may participate in non-linear pathways or combinatorial regulation.
- These prioritized genes can be mapped to GO/KEGG databases for pathway enrichment and mechanism hypothesis generation.

In a real project, this table would feed directly into:
- wet-lab validation design
- pathway enrichment analysis
- candidate diagnostic panel development

## Summary

| Perspective | Ranking Signal | Strength | Blind Spot |
|------------|----------------|----------|------------|
| Biostatistics | p-value / fold change | Rigor, uncertainty quantification | Misses non-linear interactions |
| Machine Learning | SHAP / feature importance | Predictive relevance | Can overstate spurious patterns without validation |
| **Integrated View** | SHAP + significance + pathway analysis | **Best for biomarker discovery** | Requires more careful workflow design |

This is the central message of the project: **statistics tells us whether a signal is credible; machine learning tells us whether it is useful for prediction; biological interpretation tells us whether it matters.**